# MuonClip angular/radial RG — multi-seed aggregate

This exploratory notebook uses the same strict multi-seed engine as the canonical Angular WeightWatcher notebook. It analyzes every completed training seed with saved **initial**, **best**, and **final** checkpoints, then emphasizes one matrix selected by `RG_MATRIX_NAME`.

The scientific replicate is the training seed. Random-null draws are never counted as seeds. Cross-seed summaries use the mean with **95% Student-t confidence intervals**, retain individual seed points, compare directly with the matched Haar/Stiefel random-angular baseline, and generate both full-y and **zoomed-y** figures.


## Fit contract

The continuous projective angular values are passed directly to `powerlaw.Fit(values, discrete=False, verbose=False)` with **no `xmin` and no `xmax` supplied**. The package selects $x_{min}$ by its MLE/KS procedure and keeps the far end of the observed tail. Upper endpoint atoms are counted separately rather than converted into giant artificial values.

For every seed and flow we compare alpha, $x_{min}$, KS $D$, tail length, tail population, endpoint atoms, and empirical spectrum/tail distances against the matched random-angular null.


## Papermill usage

Use a campaign-level `RESULTS_ROOT`; normally leave `RUN_DIR` unset. Blank `ANGULAR_SEEDS` auto-discovers all completed seeds. A value such as `1337,2027,31415,271828` restricts the analysis to exactly those seeds.

```bash
export RG_OPTIMIZERS_ROOT=/tmp/rg_optimizers
export RUNROOT=/tmp/<campaign-run-root>
export RESULTS_ROOT="$RUNROOT/results"
export TARGET_OPTIMIZER=muon_clip
unset RUN_DIR INITIAL_CHECKPOINT_PATH BEST_CHECKPOINT_PATH FINAL_CHECKPOINT_PATH
export ANGULAR_SEEDS=""
export RG_MATRIX_NAME=L00_W_Q
export ANGULAR_N_NULL=100
export ANGULAR_ENDPOINT_TOL=1e-10
export ANGULAR_SHOW_PLOTS=0
papermill baseline/nanogpt_one_head/notebooks/angular/muonclip_angular_radial_rg.ipynb /tmp/angular_rg_multiseed.out.ipynb
```


In [ ]:
import os
TARGET_OPTIMIZER = os.environ.get("TARGET_OPTIMIZER", os.environ.get("OPTIMIZER_NAME", "muon_clip"))
RESULTS_ROOT = os.environ.get("RESULTS_ROOT", "")
RUNROOT = os.environ.get("RUNROOT", "")
RUN_DIR = os.environ.get("RUN_DIR", "")
ANGULAR_SEEDS = os.environ.get("ANGULAR_SEEDS", "")
ANGULAR_OUTPUT_DIR = os.environ.get("ANGULAR_OUTPUT_DIR", "")
ANGULAR_N_NULL = int(os.environ.get("ANGULAR_N_NULL", "100"))
ANGULAR_N_ENTRY_NULL = int(os.environ.get("ANGULAR_N_ENTRY_NULL", "24"))
ANGULAR_MIN_TAIL = int(os.environ.get("ANGULAR_MIN_TAIL", "20"))
ANGULAR_NULL_SEED = int(os.environ.get("ANGULAR_NULL_SEED", "91337"))
ANGULAR_ENDPOINT_TOL = float(os.environ.get("ANGULAR_ENDPOINT_TOL", "1e-10"))
ANGULAR_SHOW_PLOTS = os.environ.get("ANGULAR_SHOW_PLOTS", "0")
RG_MATRIX_NAME = os.environ.get("RG_MATRIX_NAME", "L00_W_Q")


In [ ]:
from pathlib import Path
import sys

def _as_bool(value):
    if isinstance(value, bool): return value
    return str(value).strip().lower() not in {"0", "false", "no", "off"}

def _none_if_blank(value):
    text = str(value).strip() if value is not None else ""
    return text or None

def find_experiment_root():
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents): candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file(): return candidate
    raise FileNotFoundError("Set RG_OPTIMIZERS_ROOT or launch from rg_optimizers")

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
os.environ["ANGULAR_ENDPOINT_TOL"] = str(ANGULAR_ENDPOINT_TOL)
from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_multiseed import run_multiseed_analysis
CONFIG = AnalysisConfig(optimizer=str(TARGET_OPTIMIZER).lower(), results_root=_none_if_blank(RESULTS_ROOT), runroot=_none_if_blank(RUNROOT), run_dir=_none_if_blank(RUN_DIR), output_dir=_none_if_blank(ANGULAR_OUTPUT_DIR), angular_nulls=int(ANGULAR_N_NULL), entry_nulls=int(ANGULAR_N_ENTRY_NULL), min_tail=int(ANGULAR_MIN_TAIL), null_seed=int(ANGULAR_NULL_SEED), show_plots=_as_bool(ANGULAR_SHOW_PLOTS))
print("ANGULAR_SEEDS =", ANGULAR_SEEDS or "<auto-discover all completed seeds>")
print("RG_MATRIX_NAME =", RG_MATRIX_NAME)


In [ ]:
from IPython.display import display
SEED_RESULTS, CROSS_SEED, MANIFEST = run_multiseed_analysis(CONFIG, seed_spec=str(ANGULAR_SEEDS))
print("Seeds analyzed:", MANIFEST["seeds"])
print("Error bars:", MANIFEST["error_bar_contract"])
print("Output directory:", MANIFEST["output_dir"])
display(SEED_RESULTS[SEED_RESULTS["matrix_name"] == str(RG_MATRIX_NAME)])
display(CROSS_SEED[CROSS_SEED["matrix_name"] == str(RG_MATRIX_NAME)])


In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path
for metric in ("alpha", "tail_decades", "D", "xmin"):
    display(Markdown(f"## Cross-seed {metric.replace('_', ' ')}"))
    for suffix in ("full_y", "zoom_y"):
        for path in [Path(p) for p in MANIFEST["plots"] if f"cross_seed_{metric}_" in Path(p).name and suffix in Path(p).name]:
            display(Markdown(f"### `{path.name}`"))
            display(Image(filename=str(path), width=1100))


## Interpretation

Individual dots are seed-level results. Error bars are mean $\pm$ 95% Student-t confidence intervals across training seeds. The random comparison is the cross-seed distribution of the matched random-null median. Full-y and zoom-y plots are both retained.

The zoomed plot is for visibility only: it does not truncate the angular sample, alter `powerlaw.Fit`, remove extreme values, or change the null. The inference question is whether a trained angular tail is reproducibly different from the random-angular baseline across seeds.
